# Advanced Patterns in LangGraph

## Tutorials Covered:
1. Hierarchical Task Decomposition
2. Self-Correction Mechanisms
3. Collaborative Agents
4. Adaptive Learning
5. Future of LangGraph

## 1. Hierarchical Task Decomposition

Learning objectives:
- Break down complex tasks into subtasks
- Implement hierarchical planning systems
- Design task coordination mechanisms

In [ ]:
# Tutorial 1: Hierarchical Task Decomposition

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import json

print("=== Hierarchical Task Decomposition in LangGraph ===\n")

# Define state for hierarchical task decomposition
class HierarchicalTaskState(TypedDict):
    main_task: str
    subtasks: List[dict]
    completed_subtasks: List[dict]
    current_subtask_index: int
    task_hierarchy: dict
    overall_progress: float
    task_results: List[str]

# Define nodes for hierarchical task decomposition

def task_planner_node(state):
    print(f"Planning main task: {state['main_task']}")
    
    # Break down the main task into subtasks
    main_task_lower = state['main_task'].lower()
    
    if "website" in main_task_lower:
        subtasks = [
            {"id": 1, "description": "Plan website structure", "status": "pending", "dependencies": []},
            {"id": 2, "description": "Design user interface", "status": "pending", "dependencies": [1]},
            {"id": 3, "description": "Develop frontend components", "status": "pending", "dependencies": [2]},
            {"id": 4, "description": "Build backend services", "status": "pending", "dependencies": [1]},
            {"id": 5, "description": "Integrate frontend and backend", "status": "pending", "dependencies": [3, 4]},
            {"id": 6, "description": "Test website functionality", "status": "pending", "dependencies": [5]},
            {"id": 7, "description": "Deploy website", "status": "pending", "dependencies": [6]}
        ]
    elif "research" in main_task_lower or "analyze" in main_task_lower:
        subtasks = [
            {"id": 1, "description": "Define research questions", "status": "pending", "dependencies": []},
            {"id": 2, "description": "Collect relevant data", "status": "pending", "dependencies": [1]},
            {"id": 3, "description": "Clean and preprocess data", "status": "pending", "dependencies": [2]},
            {"id": 4, "description": "Perform exploratory analysis", "status": "pending", "dependencies": [3]},
            {"id": 5, "description": "Apply analytical models", "status": "pending", "dependencies": [4]},
            {"id": 6, "description": "Interpret results", "status": "pending", "dependencies": [5]},
            {"id": 7, "description": "Write research report", "status": "pending", "dependencies": [6]}
        ]
    else:
        subtasks = [
            {"id": 1, "description": "Analyze task requirements", "status": "pending", "dependencies": []},
            {"id": 2, "description": "Plan approach", "status": "pending", "dependencies": [1]},
            {"id": 3, "description": "Execute primary action", "status": "pending", "dependencies": [2]},
            {"id": 4, "description": "Validate results", "status": "pending", "dependencies": [3]},
            {"id": 5, "description": "Document outcomes", "status": "pending", "dependencies": [4]}
        ]
    
    # Create task hierarchy
    hierarchy = {
        "main_task": state['main_task'],
        "subtasks": subtasks,
        "total_subtasks": len(subtasks)
    }
    
    return {
        "subtasks": subtasks,
        "current_subtask_index": 0,
        "task_hierarchy": hierarchy,
        "overall_progress": 0.0,
        "task_results": [f"Planned {len(subtasks)} subtasks for '{state['main_task']}'"]
    }

def subtask_executor_node(state):
    # Find next pending subtask that has satisfied dependencies
    subtasks = state['subtasks']
    completed_ids = [t['id'] for t in state['completed_subtasks']]
    
    next_subtask = None
    for subtask in subtasks:
        if subtask['status'] == 'pending':
            # Check if all dependencies are completed
            dependencies_met = all(dep in completed_ids for dep in subtask.get('dependencies', []))
            if dependencies_met:
                next_subtask = subtask
                break
    
    if next_subtask:
        print(f"Executing subtask {next_subtask['id']}: {next_subtask['description']}")
        
        # Simulate subtask execution
        import time
        time.sleep(0.1)  # Simulate work
        
        # Mark as completed
        completed_subtask = {
            **next_subtask,
            "status": "completed",
            "result": f"Result of {next_subtask['description']}",
            "timestamp": time.time()
        }
        
        # Update subtask in the list
        updated_subtasks = []
        for st in subtasks:
            if st['id'] == next_subtask['id']:
                updated_subtasks.append(completed_subtask)
            else:
                updated_subtasks.append(st)
        
        completed_list = state['completed_subtasks'] + [completed_subtask]
        progress = len(completed_list) / len(subtasks) * 100
        
        return {
            "subtasks": updated_subtasks,
            "completed_subtasks": completed_list,
            "overall_progress": progress,
            "task_results": state['task_results'] + [f"Completed subtask {next_subtask['id']}: {next_subtask['description'][:30]}..."]
        }
    else:
        # No more subtasks to execute
        return {
            "overall_progress": 100.0,
            "task_results": state['task_results'] + ["All subtasks completed"]
        }

def progress_tracker_node(state):
    print(f"Tracking progress: {state['overall_progress']:.1f}%")
    
    # Check if all subtasks are completed
    all_completed = len(state['completed_subtasks']) >= len(state['subtasks'])
    
    return {
        "task_results": state['task_results'] + [f"Progress: {state['overall_progress']:.1f}%"]
    }

def task_assembler_node(state):
    print(f"Assembling results for main task: {state['main_task']}")
    
    # Combine all subtask results
    results_summary = f"Main task '{state['main_task']}' completed with {len(state['completed_subtasks'])} subtasks. "
    results_summary += "Subtasks completed: " + ", ".join([st['description'][:20] for st in state['completed_subtasks']])
    
    return {
        "task_results": state['task_results'] + [results_summary]
    }

# Routing function to determine next action
def task_completion_router(state):
    all_completed = len(state['completed_subtasks']) >= len(state['subtasks'])
    print(f"Routing based on completion: {not all_completed}")
    return "continue" if not all_completed else "complete"

print("1. HIERARCHICAL TASK DECOMPOSITION breaks complex tasks into manageable subtasks")
print("   Tasks are organized in dependency-aware hierarchies\n")

# Create the hierarchical task decomposition graph
builder = StateGraph(HierarchicalTaskState)

# Add nodes
builder.add_node("task_planner", task_planner_node)
builder.add_node("subtask_executor", subtask_executor_node)
builder.add_node("progress_tracker", progress_tracker_node)
builder.add_node("task_assembler", task_assembler_node)

# Set up the flow
builder.add_edge("__start__", "task_planner")

# Add conditional edges for iterative execution
builder.add_conditional_edges(
    "task_planner",
    task_completion_router,
    {
        "continue": "subtask_executor",
        "complete": "task_assembler"
    }
)

builder.add_conditional_edges(
    "subtask_executor",
    task_completion_router,
    {
        "continue": "progress_tracker",
        "complete": "task_assembler"
    }
)

builder.add_edge("progress_tracker", "subtask_executor")
builder.add_edge("task_assembler", "__end__")

# Compile the hierarchical task system
hierarchical_task_system = builder.compile()

print("2. Executing the hierarchical task decomposition system with different tasks:\n")

# Test the hierarchical task system with different tasks
tasks = [
    "Build a responsive e-commerce website",
    "Analyze customer satisfaction survey data",
    "Create a comprehensive marketing plan"
]

for i, task in enumerate(tasks):
    print(f"--- Task {i+1}: {task} ---")
    result = hierarchical_task_system.invoke({
        "main_task": task,
        "subtasks": [],
        "completed_subtasks": [],
        "current_subtask_index": 0,
        "task_hierarchy": {},
        "overall_progress": 0.0,
        "task_results": []
    })
    print(f"Overall progress: {result['overall_progress']:.1f}%")
    print(f"Subtasks completed: {len(result['completed_subtasks'])}")
    print(f"Task results: {result['task_results'][-2:]}")
    print()

print("3. Key concepts of hierarchical task decomposition:")
print("   - Complex tasks are decomposed into smaller, manageable subtasks")
print("   - Dependencies between subtasks are explicitly modeled")
print("   - Progress tracking enables monitoring of task completion")
print("   - Results from subtasks are aggregated to complete the main task")

## 2. Self-Correction Mechanisms

Learning objectives:
- Build agents that can detect and fix their own errors
- Implement validation and verification systems
- Design auto-correction workflows

In [ ]:
# Tutorial 2: Self-Correction Mechanisms

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import json
import re

print("=== Self-Correction Mechanisms in LangGraph ===\n")

# Define state for self-correction system
class SelfCorrectionState(TypedDict):
    original_task: str
    proposed_solution: str
    validation_result: dict
    correction_attempts: int
    corrected_solution: str
    error_log: List[str]
    solution_quality: float

# Define nodes for self-correction system

def solution_generator_node(state):
    print(f"Generating solution for task: {state['original_task']}")
    
    # Generate a proposed solution (with potential errors)
    task_lower = state['original_task'].lower()
    
    if "math" in task_lower or any(op in task_lower for op in ["+", "-", "*", "/"]):
        # Intentionally introduce some errors in math solutions
        import random
        if random.random() < 0.3:  # 30% chance of error
            solution = f"MATH SOLUTION: The incorrect answer is {random.randint(1, 100)} (intentional error for demo)"
        else:
            solution = f"MATH SOLUTION: The correct approach involves standard mathematical operations"
    elif "write" in task_lower or "essay" in task_lower:
        # Intentionally introduce some errors in writing
        solution = f"ESSAY SOLUTION: This is a sample response to the task '{state['original_task']}'. It contains several well-structured paragraphs with proper introduction, body, and conclusion. However, there might be some inconsistencies or logical gaps that need to be identified through validation."
    else:
        solution = f"GENERAL SOLUTION: To address '{state['original_task']}', we should consider multiple approaches including A, B, and C. The implementation should follow best practices and consider edge cases."
    
    return {
        "proposed_solution": solution,
        "correction_attempts": 0,
        "error_log": [f"Generated initial solution with {len(solution)} characters"]
    }

def validator_node(state):
    print(f"Validating solution: {state['proposed_solution'][:50]}...")
    
    # Validate the proposed solution
    validation_issues = []
    
    # Check for common issues
    if len(state['proposed_solution']) < 20:
        validation_issues.append("Solution is too short")
    
    if "incorrect" in state['proposed_solution'].lower():
        validation_issues.append("Solution contains explicit error indication")
    
    if not re.search(r'[.!?]$', state['proposed_solution']):
        validation_issues.append("Solution doesn't end with proper punctuation")
    
    if "might be" in state['proposed_solution'].lower() or "could be" in state['proposed_solution'].lower():
        validation_issues.append("Solution contains uncertain language")
    
    # Calculate quality score
    quality_score = 1.0 - (len(validation_issues) * 0.1)  # Lower quality with more issues
    quality_score = max(0.0, quality_score)  # Ensure non-negative
    
    validation_result = {
        "issues_found": len(validation_issues) > 0,
        "issues": validation_issues,
        "quality_score": quality_score,
        "timestamp": time.time()
    }
    
    return {
        "validation_result": validation_result,
        "solution_quality": quality_score,
        "error_log": state['error_log'] + [f"Found {len(validation_issues)} issues during validation"]
    }

def correction_node(state):
    print(f"Correcting solution due to issues: {state['validation_result']['issues']}")
    
    import time
    time.sleep(0.1)  # Simulate correction time
    
    # Apply corrections based on validation issues
    current_solution = state['proposed_solution']
    correction_attempts = state['correction_attempts'] + 1
    
    corrected_solution = current_solution
    
    # Fix issues based on validation results
    for issue in state['validation_result']['issues']:
        if "too short" in issue:
            corrected_solution += " This has been expanded with additional relevant information to meet length requirements."
        elif "explicit error" in issue:
            corrected_solution = corrected_solution.replace("incorrect", "correct")
        elif "punctuation" in issue:
            corrected_solution += "."
        elif "uncertain language" in issue:
            corrected_solution = corrected_solution.replace("might be", "is").replace("could be", "is")
    
    return {
        "corrected_solution": corrected_solution,
        "proposed_solution": corrected_solution,  # Update proposed solution for re-validation
        "correction_attempts": correction_attempts,
        "error_log": state['error_log'] + [f"Applied correction attempt #{correction_attempts}"]
    }

def quality_assessor_node(state):
    print(f"Assessing final solution quality: {state['solution_quality']:.2f}")
    
    # Determine if the solution is acceptable
    is_acceptable = state['solution_quality'] >= 0.7
    
    final_solution = state['proposed_solution'] if not state['corrected_solution'] else state['corrected_solution']
    
    return {
        "corrected_solution": final_solution,
        "error_log": state['error_log'] + [f"Quality assessment: Acceptable = {is_acceptable}"]
    }

# Import time for timestamp usage
import time

# Routing function to determine next action
def correction_router(state):
    needs_correction = state['validation_result'].get('issues_found', False)
    attempts_remaining = state['correction_attempts'] < 3  # Max 3 correction attempts
    
    print(f"Routing: needs_correction={needs_correction}, attempts_remaining={attempts_remaining}")
    
    if needs_correction and attempts_remaining:
        return 'correction'
    else:
        return 'assessment'

print("1. SELF-CORRECTION MECHANISMS enable agents to validate and fix their own outputs")
print("   Solutions are checked against validation criteria and corrected if needed\n")

# Create the self-correction graph
builder = StateGraph(SelfCorrectionState)

# Add nodes
builder.add_node("solution_generator", solution_generator_node)
builder.add_node("validator", validator_node)
builder.add_node("correction", correction_node)
builder.add_node("quality_assessor", quality_assessor_node)

# Set up the flow
builder.add_edge("__start__", "solution_generator")
builder.add_edge("solution_generator", "validator")

# Add conditional edges for correction loop
builder.add_conditional_edges(
    "validator",
    correction_router,
    {
        "correction": "correction",
        "assessment": "quality_assessor"
    }
)

builder.add_edge("correction", "validator")  # Loop back to validate the correction
builder.add_edge("quality_assessor", "__end__")

# Compile the self-correction system
self_correction_system = builder.compile()

print("2. Executing the self-correction system with different tasks:\n")

# Test the self-correction system with different tasks
tasks = [
    "Solve the math equation: 25 + 37",
    "Write an essay about the importance of renewable energy",
    "Explain how machine learning algorithms work"
]

for i, task in enumerate(tasks):
    print(f"--- Task {i+1}: {task} ---")
    result = self_correction_system.invoke({
        "original_task": task,
        "proposed_solution": "",
        "validation_result": {},
        "correction_attempts": 0,
        "corrected_solution": "",
        "error_log": [],
        "solution_quality": 0.0
    })
    print(f"Original task: {result['original_task']}")
    print(f"Correction attempts: {result['correction_attempts']}")
    print(f"Solution quality: {result['solution_quality']:.2f}")
    print(f"Error log: {result['error_log'][-2:]}")
    print(f"Final solution (first 100 chars): {result['corrected_solution'][:100]}...")
    print()

print("3. Key concepts of self-correction mechanisms:")
print("   - Solutions are validated against predefined criteria")
print("   - Errors are identified and logged automatically")
print("   - Correction attempts refine the solution iteratively")
print("   - Quality assessment determines if the solution is acceptable")

## 3. Collaborative Agents

Learning objectives:
- Enable multiple agents working together on a shared goal
- Implement inter-agent communication protocols
- Design collaboration frameworks

In [ ]:
# Tutorial 3: Collaborative Agents

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import json

print("=== Collaborative Agents in LangGraph ===\n")

# Define state for collaborative agents system
class CollaborativeAgentsState(TypedDict):
    main_task: str
    agent_team: List[dict]
    agent_contributions: List[dict]
    collective_solution: str
    collaboration_history: List[str]
    agent_performance: dict
    final_decision: str

# Define different specialized agents

class Agent:
    def __init__(self, name: str, specialty: str, expertise_level: float):
        self.name = name
        self.specialty = specialty
        self.expertise_level = expertise_level
    
    def contribute(self, task: str, context: dict) -> dict:
        # Simulate contribution based on specialty
        import random
        
        if self.specialty == "research":
            contribution = f"RESEARCH AGENT {self.name}: After thorough analysis of '{task}', I found that key factors include {random.choice(['market trends', 'user behavior', 'technical feasibility'])}. My research suggests {random.choice(['approach A', 'approach B', 'approach C'])} would be most effective."
        elif self.specialty == "analysis":
            contribution = f"ANALYSIS AGENT {self.name}: Evaluating the task '{task}', I've identified critical components: {random.choice(['cost-benefit', 'risk assessment', 'resource allocation'])}. The data indicates {random.choice(['positive', 'neutral', 'cautious')]}) outlook."
        elif self.specialty == "strategy":
            contribution = f"STRATEGY AGENT {self.name}: For '{task}', I recommend a phased approach focusing on {random.choice(['innovation', 'efficiency', 'scalability'])}. Implementation should prioritize {random.choice(['speed', 'quality', 'cost-effectiveness'])}."
        elif self.specialty == "validation":
            contribution = f"VALIDATION AGENT {self.name}: I've reviewed the proposed approaches for '{task}'. Potential issues include {random.choice(['feasibility concerns', 'resource constraints', 'timeline risks'])}. I recommend {random.choice(['additional research', 'alternative approach', 'risk mitigation'])}."
        else:
            contribution = f"GENERAL AGENT {self.name}: Regarding '{task}', my perspective focuses on {random.choice(['practical implementation', 'theoretical foundation', 'best practices'])}. Key considerations are {random.choice(['budget', 'timeline', 'quality'])}."
        
        # Simulate performance based on expertise level
        effectiveness = min(1.0, self.expertise_level + random.uniform(-0.1, 0.1))
        
        return {
            "agent_name": self.name,
            "specialty": self.specialty,
            "contribution": contribution,
            "effectiveness": effectiveness,
            "timestamp": time.time()
        }

# Define nodes for collaborative agents

def team_assembly_node(state):
    print(f"Assembling agent team for task: {state['main_task']}")
    
    # Create specialized agents
    agents = [
        Agent("ResearcherAlpha", "research", 0.9),
        Agent("AnalystBeta", "analysis", 0.85),
        Agent("StrategistGamma", "strategy", 0.88),
        Agent("ValidatorDelta", "validation", 0.82)
    ]
    
    agent_team_info = [{"name": agent.name, "specialty": agent.specialty, "expertise": agent.expertise_level} for agent in agents]
    
    return {
        "agent_team": agent_team_info,
        "collaboration_history": [f"Assembled team of {len(agents)} agents for '{state['main_task']}'"],
        "agent_performance": {agent.name: 0.0 for agent in agents}
    }

def research_phase_node(state):
    print(f"Starting research phase")
    
    # Find research agents
    research_agents = [agent for agent in state['agent_team'] if agent['specialty'] == 'research']
    
    contributions = []
    for agent_info in research_agents:
        agent = Agent(agent_info['name'], agent_info['specialty'], agent_info['expertise'])
        contribution = agent.contribute(state['main_task'], {})
        contributions.append(contribution)
    
    return {
        "agent_contributions": contributions,
        "collaboration_history": state['collaboration_history'] + [f"Completed research phase with {len(research_agents)} research agents"]
    }

def analysis_phase_node(state):
    print(f"Starting analysis phase")
    
    # Get previous contributions as context
    previous_contributions = [c['contribution'] for c in state['agent_contributions']]
    context = {"previous_analysis": previous_contributions}
    
    # Find analysis agents
    analysis_agents = [agent for agent in state['agent_team'] if agent['specialty'] == 'analysis']
    
    new_contributions = []
    for agent_info in analysis_agents:
        agent = Agent(agent_info['name'], agent_info['specialty'], agent_info['expertise'])
        contribution = agent.contribute(state['main_task'], context)
        new_contributions.append(contribution)
    
    return {
        "agent_contributions": state['agent_contributions'] + new_contributions,
        "collaboration_history": state['collaboration_history'] + [f"Completed analysis phase with {len(analysis_agents)} analysis agents"]
    }

def strategy_phase_node(state):
    print(f"Starting strategy phase")
    
    # Get all previous contributions as context
    previous_contributions = [c['contribution'] for c in state['agent_contributions']]
    context = {"all_previous_analysis": previous_contributions}
    
    # Find strategy agents
    strategy_agents = [agent for agent in state['agent_team'] if agent['specialty'] == 'strategy']
    
    new_contributions = []
    for agent_info in strategy_agents:
        agent = Agent(agent_info['name'], agent_info['specialty'], agent_info['expertise'])
        contribution = agent.contribute(state['main_task'], context)
        new_contributions.append(contribution)
    
    return {
        "agent_contributions": state['agent_contributions'] + new_contributions,
        "collaboration_history": state['collaboration_history'] + [f"Completed strategy phase with {len(strategy_agents)} strategy agents"]
    }

def validation_phase_node(state):
    print(f"Starting validation phase")
    
    # Get all previous contributions as context
    previous_contributions = [c['contribution'] for c in state['agent_contributions']]
    context = {"proposed_solution": previous_contributions}
    
    # Find validation agents
    validation_agents = [agent for agent in state['agent_team'] if agent['specialty'] == 'validation']
    
    new_contributions = []
    for agent_info in validation_agents:
        agent = Agent(agent_info['name'], agent_info['specialty'], agent_info['expertise'])
        contribution = agent.contribute(state['main_task'], context)
        new_contributions.append(contribution)
    
    return {
        "agent_contributions": state['agent_contributions'] + new_contributions,
        "collaboration_history": state['collaboration_history'] + [f"Completed validation phase with {len(validation_agents)} validation agents"]
    }

def solution_synthesis_node(state):
    print(f"Synthesizing collective solution")
    
    # Aggregate all contributions into a coherent solution
    all_contributions = [c['contribution'] for c in state['agent_contributions']]
    
    # Calculate average effectiveness
    avg_effectiveness = sum(c['effectiveness'] for c in state['agent_contributions']) / len(state['agent_contributions']) if state['agent_contributions'] else 0
    
    # Generate synthesized solution
    synthesized_solution = f"COLLECTIVE SOLUTION: Based on collaborative input from {len(state['agent_team'])} specialized agents:\n"
    for contrib in all_contributions:
        # Extract key points from each contribution
        key_elements = contrib.split('. ')[:2]  # Take first two sentences
        synthesized_solution += f"- {key_elements[0][:100]}...\n"
    
    synthesized_solution += f"\nOverall confidence in this solution: {avg_effectiveness*100:.1f}%"
    
    return {
        "collective_solution": synthesized_solution,
        "final_decision": f"Recommended approach for '{state['main_task']}' based on agent collaboration",
        "collaboration_history": state['collaboration_history'] + [f"Synthesized solution with avg effectiveness {avg_effectiveness:.2f}"]
    }

import time

print("1. COLLABORATIVE AGENTS bring together specialized agents with different expertise")
print("   Agents work together to produce better solutions than any single agent could\n")

# Create the collaborative agents graph
builder = StateGraph(CollaborativeAgentsState)

# Add nodes
builder.add_node("team_assembly", team_assembly_node)
builder.add_node("research_phase", research_phase_node)
builder.add_node("analysis_phase", analysis_phase_node)
builder.add_node("strategy_phase", strategy_phase_node)
builder.add_node("validation_phase", validation_phase_node)
builder.add_node("solution_synthesis", solution_synthesis_node)

# Set up the flow
builder.add_edge("__start__", "team_assembly")
builder.add_edge("team_assembly", "research_phase")
builder.add_edge("research_phase", "analysis_phase")
builder.add_edge("analysis_phase", "strategy_phase")
builder.add_edge("strategy_phase", "validation_phase")
builder.add_edge("validation_phase", "solution_synthesis")
builder.add_edge("solution_synthesis", "__end__")

# Compile the collaborative agents system
collaborative_agents_system = builder.compile()

print("2. Executing the collaborative agents system with different tasks:\n")

# Test the collaborative agents system with different tasks
tasks = [
    "Develop a strategy for increasing customer engagement",
    "Design an AI-powered recommendation system",
    "Create a plan for sustainable business growth"
]

for i, task in enumerate(tasks):
    print(f"--- Task {i+1}: {task} ---")
    result = collaborative_agents_system.invoke({
        "main_task": task,
        "agent_team": [],
        "agent_contributions": [],
        "collective_solution": "",
        "collaboration_history": [],
        "agent_performance": {},
        "final_decision": ""
    })
    print(f"Team size: {len(result['agent_team'])}")
    print(f"Contributions received: {len(result['agent_contributions'])}")
    print(f"Collaboration history: {result['collaboration_history'][-2:]}")
    print(f"Final decision: {result['final_decision']}")
    print(f"Collective solution preview: {result['collective_solution'][:200]}...")
    print()

print("3. Key concepts of collaborative agents:")
print("   - Specialized agents contribute their domain expertise")
print("   - Contributions are aggregated into comprehensive solutions")
print("   - Phased approach allows for building upon previous insights")
print("   - Collective intelligence often surpasses individual capabilities")

## 4. Adaptive Learning

Learning objectives:
- Create agents that improve over time based on interactions
- Implement learning from experience mechanisms
- Design feedback integration systems

In [ ]:
# Tutorial 4: Adaptive Learning

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import json
import time

print("=== Adaptive Learning in LangGraph ===\n")

# Define state for adaptive learning system
class AdaptiveLearningState(TypedDict):
    task: str
    initial_approach: str
    learning_history: List[dict]
    performance_metrics: dict
    adaptive_strategy: str
    improved_approach: str
    learning_rate: float

# Define nodes for adaptive learning system

def initial_problem_solver_node(state):
    print(f"Solving initial problem: {state['task']}")
    
    # Generate an initial approach to the problem
    import random
    approaches = [
        "Apply standard methodology",
        "Use heuristic approach",
        "Try experimental method",
        "Follow best practices guide",
        "Implement systematic solution"
    ]
    
    initial_approach = random.choice(approaches)
    
    # Simulate performance of initial approach
    initial_performance = {
        "accuracy": random.uniform(0.5, 0.8),
        "efficiency": random.uniform(0.4, 0.7),
        "robustness": random.uniform(0.5, 0.75),
        "time_taken": random.uniform(1.0, 3.0)
    }
    
    learning_record = {
        "attempt": 1,
        "approach": initial_approach,
        "performance": initial_performance,
        "feedback": "Initial baseline performance recorded",
        "timestamp": time.time()
    }
    
    return {
        "initial_approach": initial_approach,
        "learning_history": [learning_record],
        "performance_metrics": initial_performance,
        "learning_rate": 0.1
    }

def performance_evaluator_node(state):
    print(f"Evaluating performance of approach: {state['initial_approach']}")
    
    # Calculate overall performance score
    perf = state['performance_metrics']
    overall_score = (perf['accuracy'] + perf['efficiency'] + perf['robustness']) / 3
    
    # Determine areas for improvement
    areas_for_improvement = []
    if perf['accuracy'] < 0.7:
        areas_for_improvement.append("accuracy")
    if perf['efficiency'] < 0.6:
        areas_for_improvement.append("efficiency")
    if perf['robustness'] < 0.7:
        areas_for_improvement.append("robustness")
    
    evaluation_feedback = {
        "overall_score": overall_score,
        "areas_for_improvement": areas_for_improvement,
        "strengths": [metric for metric, value in perf.items() if value > 0.7 and metric != 'time_taken']
    }
    
    # Update learning history
    latest_record = state['learning_history'][-1]
    updated_record = {**latest_record, "evaluation": evaluation_feedback}
    
    return {
        "learning_history": state['learning_history'][:-1] + [updated_record],
        "adaptive_strategy": f"Focus on improving {', '.join(areas_for_improvement) if areas_for_improvement else 'maintain current approach'}"
    }

def learning_optimizer_node(state):
    print(f"Learning from previous attempts to optimize approach")
    
    # Analyze learning history to inform improvements
    history = state['learning_history']
    
    # Identify patterns in what worked and what didn't
    if len(history) > 1:
        # Compare performance between attempts
        prev_perf = history[-2]['performance']
        curr_perf = history[-1]['performance']
        
        improvement_directions = []
        for metric in ['accuracy', 'efficiency', 'robustness']:
            if curr_perf[metric] > prev_perf[metric]:
                improvement_directions.append(f"Increased focus on {metric}-related techniques")
            else:
                improvement_directions.append(f"Adjust {metric}-improving strategies")
    else:
        improvement_directions = ["Establish baseline optimization heuristics"]
    
    # Generate improved approach based on learning
    base_approaches = ["standard methodology", "heuristic approach", "experimental method", "best practices", "systematic solution"]
    learned_approach = f"Adapted {state['initial_approach'].lower()} with focus on {', '.join(history[-1]['evaluation']['areas_for_improvement'][:2])} and {improvement_directions[0].split()[0]} optimization"
    
    return {
        "improved_approach": learned_approach,
        "adaptive_strategy": f"Apply lessons from {len(history)} previous attempt(s): {'; '.join(improvement_directions[:2])}"
    }

def iterative_refinement_node(state):
    print(f"Refining approach based on adaptive learning")
    
    # Simulate applying the improved approach and measuring performance
    import random
    
    # Slightly improved performance based on learning
    current_attempt = len(state['learning_history']) + 1
    improvement_factor = 1 + (state['learning_rate'] * current_attempt * 0.1)
    
    base_performance = state['performance_metrics']
    refined_performance = {
        key: min(1.0, value * improvement_factor) if key != 'time_taken' else value * 0.9  # Less time taken = more efficient
        for key, value in base_performance.items()
    }
    
    # Adjust based on specific learning
    for area in state['learning_history'][-1]['evaluation']['areas_for_improvement']:
        if area in refined_performance:
            refined_performance[area] = min(1.0, refined_performance[area] + 0.1)
    
    learning_record = {
        "attempt": current_attempt,
        "approach": state['improved_approach'],
        "performance": refined_performance,
        "feedback": f"Performance improved from previous attempt, especially in {', '.join(state['learning_history'][-1]['evaluation']['areas_for_improvement'])}",
        "timestamp": time.time()
    }
    
    # Calculate if we've reached satisfactory performance
    overall_score = (refined_performance['accuracy'] + refined_performance['efficiency'] + refined_performance['robustness']) / 3
    satisfaction_threshold = 0.8
    
    return {
        "performance_metrics": refined_performance,
        "learning_history": state['learning_history'] + [learning_record],
        "learning_rate": min(0.5, state['learning_rate'] + 0.05),  # Increase learning rate over time
        "adaptive_strategy": f"Current satisfaction: {overall_score:.2f}, threshold: {satisfaction_threshold}"
    }

def meta_learning_node(state):
    print(f"Applying meta-learning to generalize improvements")
    
    # Analyze the complete learning history to extract generalizable patterns
    history = state['learning_history']
    
    # Identify most effective approach modifications
    accuracy_improvements = []
    efficiency_improvements = []
    
    for record in history:
        perf = record['performance']
        if 'evaluation' in record:
            for area in record['evaluation']['areas_for_improvement']:
                if area == 'accuracy' and perf['accuracy'] > 0.7:
                    accuracy_improvements.append(record['approach'])
                elif area == 'efficiency' and perf['efficiency'] > 0.6:
                    efficiency_improvements.append(record['approach'])
    
    # Generate meta-strategy
    meta_strategy = {
        "effective_accuracy_methods": list(set(accuracy_improvements[-2:])),
        "effective_efficiency_methods": list(set(efficiency_improvements[-2:])),
        "general_principles": [
            "Iterative refinement improves performance",
            "Focus on weakest metrics first",
            "Balance between competing objectives"
        ]
    }
    
    return {
        "adaptive_strategy": f"Meta-learning insights: {json.dumps(meta_strategy, indent=2)[:200]}...",
        "improved_approach": f"Generalized approach incorporating lessons from {len(history)} learning iterations"
    }

# Routing function to determine if more learning is needed
def learning_continuation_router(state):
    # Check if performance is satisfactory
    perf = state['performance_metrics']
    overall_score = (perf['accuracy'] + perf['efficiency'] + perf['robustness']) / 3
    
    # Continue learning if below threshold and haven't exceeded max attempts
    max_attempts = 5
    current_attempts = len(state['learning_history'])
    
    should_continue = overall_score < 0.8 and current_attempts < max_attempts
    
    print(f"Routing: overall_score={overall_score:.2f}, threshold=0.8, continue={should_continue}, attempts={current_attempts}/{max_attempts}")
    
    return 'continue_learning' if should_continue else 'meta_learning'

print("1. ADAPTIVE LEARNING enables systems to improve over time based on experience")
print("   Performance is evaluated and used to refine future approaches\n")

# Create the adaptive learning graph
builder = StateGraph(AdaptiveLearningState)

# Add nodes
builder.add_node("initial_solver", initial_problem_solver_node)
builder.add_node("performance_evaluator", performance_evaluator_node)
builder.add_node("learning_optimizer", learning_optimizer_node)
builder.add_node("iterative_refinement", iterative_refinement_node)
builder.add_node("meta_learning", meta_learning_node)

# Set up the flow
builder.add_edge("__start__", "initial_solver")
builder.add_edge("initial_solver", "performance_evaluator")
builder.add_edge("performance_evaluator", "learning_optimizer")
builder.add_edge("learning_optimizer", "iterative_refinement")

# Add conditional edge for iterative learning
builder.add_conditional_edges(
    "iterative_refinement",
    learning_continuation_router,
    {
        "continue_learning": "performance_evaluator",  # Loop back to evaluate and optimize
        "meta_learning": "meta_learning"
    }
)

builder.add_edge("meta_learning", "__end__")

# Compile the adaptive learning system
adaptive_learning_system = builder.compile()

print("2. Executing the adaptive learning system with different tasks:\n")

# Test the adaptive learning system with different tasks
tasks = [
    "Optimize a recommendation algorithm",
    "Improve a classification model",
    "Enhance a search functionality"
]

for i, task in enumerate(tasks):
    print(f"--- Task {i+1}: {task} ---")
    result = adaptive_learning_system.invoke({
        "task": task,
        "initial_approach": "",
        "learning_history": [],
        "performance_metrics": {},
        "adaptive_strategy": "",
        "improved_approach": "",
        "learning_rate": 0.1
    })
    print(f"Initial approach: {result['initial_approach']}")
    print(f"Final approach: {result['improved_approach']}")
    print(f"Learning iterations: {len(result['learning_history'])}")
    print(f"Final performance: {result['performance_metrics']}")
    print(f"Adaptive strategy: {result['adaptive_strategy'][:100]}...")
    print(f"Learning rate: {result['learning_rate']:.2f}")
    print()

print("3. Key concepts of adaptive learning:")
print("   - Systems evaluate their own performance")
print("   - Insights from performance are used to improve future attempts")
print("   - Learning is iterative and builds on previous experiences")
print("   - Meta-learning extracts generalizable improvement patterns")

## 5. Future of LangGraph

Learning objectives:
- Explore trends in LangGraph development
- Understand upcoming features and capabilities
- Prepare for future advancements in the field

In [ ]:
# Tutorial 5: Future of LangGraph

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import json
import time

print("=== Future of LangGraph ===\n")

# Define state for future roadmap exploration
class FutureRoadmapState(TypedDict):
    current_trends: List[str]
    emerging_features: List[str]
    community_contributions: List[str]
    predicted_developments: List[str]
    langgraph_evolution: dict
    adoption_projections: dict

# Define nodes for exploring LangGraph's future

def trend_analysis_node(state):
    print("Analyzing current trends in LangGraph development")
    
    # Current trends in LangGraph
    trends = [
        "Integration with more LLM providers",
        "Enhanced visualization tools for graph debugging",
        "Improved performance optimizations",
        "Better support for real-time applications",
        "Advanced memory management features"
    ]
    
    return {
        "current_trends": trends,
        "community_contributions": ["Active development", "Growing ecosystem", "Regular updates"]
    }

def feature_prediction_node(state):
    print("Predicting emerging features in LangGraph")
    
    # Predicted future features
    features = [
        "Automated graph optimization",
        "Built-in A/B testing for graph variants",
        "Advanced error recovery mechanisms",
        "Cross-platform deployment tools",
        "Enhanced security features for enterprise use"
    ]
    
    return {
        "emerging_features": features
    }

def evolution_modeling_node(state):
    print("Modeling LangGraph's evolution pathway")
    
    # Model how LangGraph might evolve
    evolution_path = {
        "year_1": {
            "focus": "Stability and performance",
            "features": ["Better caching", "Improved error handling", "Enhanced documentation"]
        },
        "year_2": {
            "focus": "Scalability and enterprise features",
            "features": ["Distributed execution", "Advanced monitoring", "Security enhancements"]
        },
        "year_3": {
            "focus": "AI-assisted development",
            "features": ["Auto-graph generation", "Intelligent debugging", "Predictive optimization"]
        }
    }
    
    return {
        "langgraph_evolution": evolution_path
    }

def adoption_projection_node(state):
    print("Projecting LangGraph adoption trends")
    
    # Project adoption based on current trajectory
    projections = {
        "short_term": "Continued growth in developer adoption",
        "medium_term": "Integration into enterprise AI platforms",
        "long_term": "Potential industry standard for stateful AI applications"
    }
    
    return {
        "adoption_projections": projections
    }

def synthesis_node(state):
    print("Synthesizing insights about LangGraph's future")
    
    # Combine all insights into a comprehensive view
    synthesis = {
        "summary": f"LangGraph's future looks promising with {len(state['current_trends'])} major trends, {len(state['emerging_features'])} emerging features, and clear evolution path",
        "opportunities": ["Early adoption advantage", "Community contribution possibilities", "Specialization opportunities"],
        "challenges": ["Keeping up with rapid development", "Complexity management", "Best practices evolution"]
    }
    
    return {
        "predicted_developments": [
            "By 2025: Mature ecosystem with extensive third-party integrations",
            "By 2026: AI-assisted graph design and optimization",
            "By 2027: Industry standard for complex AI workflows"
        ]
    }

print("1. LANGGRAPH'S FUTURE encompasses evolving capabilities and growing ecosystem")
print("   The framework continues to mature with enhanced features and capabilities\n")

# Create the future roadmap graph
builder = StateGraph(FutureRoadmapState)

# Add nodes
builder.add_node("trend_analysis", trend_analysis_node)
builder.add_node("feature_prediction", feature_prediction_node)
builder.add_node("evolution_modeling", evolution_modeling_node)
builder.add_node("adoption_projection", adoption_projection_node)
builder.add_node("synthesis", synthesis_node)

# Set up the flow
builder.add_edge("__start__", "trend_analysis")
builder.add_edge("trend_analysis", "feature_prediction")
builder.add_edge("feature_prediction", "evolution_modeling")
builder.add_edge("evolution_modeling", "adoption_projection")
builder.add_edge("adoption_projection", "synthesis")
builder.add_edge("synthesis", "__end__")

# Compile the future roadmap system
future_roadmap_system = builder.compile()

print("2. Exploring LangGraph's future development roadmap:\n")

# Execute the future roadmap analysis
result = future_roadmap_system.invoke({
    "current_trends": [],
    "emerging_features": [],
    "community_contributions": [],
    "predicted_developments": [],
    "langgraph_evolution": {},
    "adoption_projections": {}
})

print(f"Current trends: {result['current_trends']}")
print(f"Emerging features: {result['emerging_features']}")
print(f"Evolution path: Years covered: {list(result['langgraph_evolution'].keys())}")
print(f"Adoption projections: {result['adoption_projections']}")
print(f"Predicted developments: {result['predicted_developments']}")

print("\n3. Key insights about LangGraph's future:")
print("   - Continued rapid development and feature expansion")
print("   - Growing ecosystem and community support")
print("   - Increasing adoption in enterprise applications")
print("   - Evolution toward AI-assisted development tools")